# Campaign-Associated Demand and Revenue Lift

This notebook evaluates whether the synthetic **Holiday Early Booking**
discount is associated with higher daily demand and net revenue, or mainly
reduces realized revenue per ticket. The primary outcome is net revenue;
demand and net revenue per ticket are secondary outcomes.

The source data are observational daily aggregates. Results are therefore
reported as **campaign-associated lift**, not randomized causal impact.

## Analysis Design

- **Treatment:** dates when Holiday Early Booking is the only active campaign.
  This removes October dates that overlap Florida Resident Email.
- **Control pool:** dates with no active campaign.
- **Matching:** same year, season, and weekend status; then nearest weekday,
  price, weather, holiday, and school-break conditions. Each treatment receives one
  nearest control.
- **Primary estimator:** paired daily lift with a seven-day moving-block
  bootstrap 95% confidence interval.
- **Sensitivity model:** log-outcome regression using all historical days,
  calendar and business controls, and seven-day HAC standard errors.
- **Pre-period check:** a 28-day lead indicator tests for a similar outcome
  shift immediately before the official campaign launch.
- **Significance level:** 5%. Net revenue is the pre-declared primary outcome.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATABASE_URL = "postgresql+psycopg:///orlando_demand_revenue"
engine = create_engine(DATABASE_URL)

daily_query = text("""
    SELECT
        daily.calendar_date,
        daily.year_number,
        daily.month_number,
        daily.day_of_week,
        daily.is_weekend,
        daily.season,
        daily.holiday_flag,
        daily.school_break_flag,
        daily.severe_weather_flag,
        daily.avg_temperature_f::double precision AS avg_temperature_f,
        daily.precipitation_in::double precision AS precipitation_in,
        daily.planned_price_multiplier::double precision
            AS planned_price_multiplier,
        daily.net_demand::double precision AS net_demand,
        daily.net_revenue::double precision AS net_revenue,
        daily.average_net_revenue_per_ticket::double precision
            AS net_revenue_per_ticket,
        daily.available_capacity::double precision AS available_capacity,
        daily.capacity_utilization::double precision AS capacity_utilization,
        count(campaign.campaign_key)::integer AS active_campaign_count,
        coalesce(bool_or(campaign.campaign_type = 'discount'), false)
            AS discount_active,
        coalesce(bool_or(campaign.campaign_type = 'paid_media'), false)
            AS paid_media_active,
        coalesce(bool_or(campaign.campaign_type = 'bundle'), false)
            AS bundle_active,
        coalesce(bool_or(campaign.campaign_type = 'email'), false)
            AS email_active
    FROM analytics.vw_daily_performance AS daily
    LEFT JOIN analytics.dim_campaign AS campaign
        ON NOT campaign.is_no_campaign
       AND daily.calendar_date BETWEEN
           campaign.start_date AND campaign.end_date
    WHERE daily.has_actuals
    GROUP BY
        daily.calendar_date,
        daily.year_number,
        daily.month_number,
        daily.day_of_week,
        daily.is_weekend,
        daily.season,
        daily.holiday_flag,
        daily.school_break_flag,
        daily.severe_weather_flag,
        daily.avg_temperature_f,
        daily.precipitation_in,
        daily.planned_price_multiplier,
        daily.net_demand,
        daily.net_revenue,
        daily.average_net_revenue_per_ticket,
        daily.available_capacity,
        daily.capacity_utilization
    ORDER BY daily.calendar_date
""")

campaign_cost_query = text("""
    WITH attributed_sales AS (
        SELECT
            purchase_date_key AS date_key,
            campaign_key,
            sum(discount_amount)::double precision
                AS attributed_discount_amount
        FROM analytics.fact_ticket_sales
        GROUP BY purchase_date_key, campaign_key
    )
    SELECT
        date_dim.calendar_date,
        campaign.campaign_code,
        delivery.spend::double precision AS marketing_spend,
        delivery.attributed_revenue::double precision AS attributed_revenue,
        coalesce(sales.attributed_discount_amount, 0)::double precision
            AS attributed_discount_amount
    FROM analytics.fact_campaign_daily AS delivery
    JOIN analytics.dim_campaign AS campaign USING (campaign_key)
    JOIN analytics.dim_date AS date_dim USING (date_key)
    LEFT JOIN attributed_sales AS sales
        ON sales.date_key = delivery.date_key
       AND sales.campaign_key = delivery.campaign_key
    WHERE campaign.campaign_code LIKE 'HOLIDAY_EARLY_%'
    ORDER BY date_dim.calendar_date
""")

with engine.connect() as connection:
    daily = pd.read_sql_query(
        daily_query, connection, parse_dates=["calendar_date"]
    )
    campaign_costs = pd.read_sql_query(
        campaign_cost_query,
        connection,
        parse_dates=["calendar_date"],
    )

assert len(daily) == 1096
assert daily["calendar_date"].is_monotonic_increasing
assert daily[["net_demand", "net_revenue"]].notna().all().all()

daily.shape, campaign_costs.shape

In [ ]:
daily["clean_discount_treatment"] = (
    daily["discount_active"]
    & daily["active_campaign_count"].eq(1)
)
daily["eligible_control"] = daily["active_campaign_count"].eq(0)
daily["trend_days"] = (
    daily["calendar_date"] - daily["calendar_date"].min()
).dt.days

treatment = daily.loc[daily["clean_discount_treatment"]].copy()
control_pool = daily.loc[daily["eligible_control"]].copy()

sample_counts = (
    daily.assign(
        sample_type=np.select(
            [
                daily["clean_discount_treatment"],
                daily["eligible_control"],
            ],
            ["clean_treatment", "no_campaign_control"],
            default="excluded_or_other_campaign",
        )
    )
    .groupby(["year_number", "sample_type"])
    .size()
    .unstack(fill_value=0)
)

assert len(treatment) == 135
assert len(control_pool) == 180
assert treatment.groupby("year_number").size().eq(45).all()
assert not treatment["email_active"].any()
assert treatment["capacity_utilization"].lt(1).all()

sample_counts

## Matching and Covariate Balance

Matching reduces observable differences between campaign and non-campaign
dates. It cannot balance variables that were not measured, so the result
remains observational. Absolute standardized mean differences below 0.10 are
commonly treated as a useful balance target rather than a proof of equivalence.

In [ ]:
MATCHES_PER_TREATMENT = 1
MATCHING_COVARIATES = (
    "holiday_flag",
    "school_break_flag",
    "severe_weather_flag",
    "planned_price_multiplier",
    "avg_temperature_f",
    "precipitation_in",
)
OUTCOMES = (
    "net_revenue",
    "net_demand",
    "net_revenue_per_ticket",
)
MATCHING_WEIGHTS = {
    covariate: 1.0 for covariate in MATCHING_COVARIATES
}
MATCHING_WEIGHTS["avg_temperature_f"] = 4.0

matching_scales = (
    pd.concat([treatment, control_pool])[list(MATCHING_COVARIATES)]
    .astype(float)
    .std(ddof=0)
    .replace(0, 1)
)


# Purpose:
# Match each treatment date to comparable no-campaign dates.
#
# Used by:
# Covariate-balance checks and paired lift estimation.
def build_matched_pairs(treated_days, candidate_controls):
    pair_records = []

    for treated_row in treated_days.itertuples(index=False):
        candidates = candidate_controls.loc[
            candidate_controls["year_number"].eq(treated_row.year_number)
            & candidate_controls["season"].eq(treated_row.season)
            & candidate_controls["is_weekend"].eq(
                treated_row.is_weekend
            )
        ].copy()

        if len(candidates) < MATCHES_PER_TREATMENT:
            raise ValueError(
                f"Insufficient controls for {treated_row.calendar_date.date()}"
            )

        squared_distance = np.zeros(len(candidates), dtype=float)
        for covariate in MATCHING_COVARIATES:
            treated_value = float(getattr(treated_row, covariate))
            candidate_value = candidates[covariate].astype(float)
            squared_distance += MATCHING_WEIGHTS[covariate] * (
                (candidate_value - treated_value) / matching_scales[covariate]
            ) ** 2

        calendar_distance = (
            candidates["calendar_date"] - treated_row.calendar_date
        ).dt.days.abs()
        weekday_distance = (
            candidates["day_of_week"] - treated_row.day_of_week
        ).abs()
        weekday_distance = np.minimum(
            weekday_distance, 7 - weekday_distance
        )
        candidates["match_distance"] = (
            np.sqrt(squared_distance)
            + 0.75 * weekday_distance
            + 0.05 * calendar_distance / 30
        )

        selected = candidates.nsmallest(
            MATCHES_PER_TREATMENT,
            ["match_distance", "calendar_date"],
        )

        for control_row in selected.itertuples(index=False):
            record = {
                "treated_date": treated_row.calendar_date,
                "control_date": control_row.calendar_date,
                "campaign_year": treated_row.year_number,
                "season": treated_row.season,
                "day_of_week": treated_row.day_of_week,
                "match_distance": control_row.match_distance,
            }
            for column in (*MATCHING_COVARIATES, *OUTCOMES):
                record[f"treated_{column}"] = getattr(treated_row, column)
                record[f"control_{column}"] = getattr(control_row, column)
            pair_records.append(record)

    return pd.DataFrame.from_records(pair_records)


# Purpose:
# Express a treatment-control difference in pooled standard-deviation units.
#
# Used by:
# Raw and post-match balance diagnostics.
def standardized_mean_difference(treated_values, control_values):
    treated_values = pd.Series(treated_values, dtype=float)
    control_values = pd.Series(control_values, dtype=float)
    pooled_standard_deviation = np.sqrt(
        (treated_values.var(ddof=1) + control_values.var(ddof=1)) / 2
    )
    if np.isclose(pooled_standard_deviation, 0):
        return 0.0 if np.isclose(
            treated_values.mean(), control_values.mean()
        ) else np.nan
    return (
        treated_values.mean() - control_values.mean()
    ) / pooled_standard_deviation


matched_pairs = build_matched_pairs(treatment, control_pool)

assert len(matched_pairs) == len(treatment) * MATCHES_PER_TREATMENT
assert matched_pairs["treated_date"].nunique() == len(treatment)

matched_pairs.head()

In [ ]:
balance_records = []
for covariate in MATCHING_COVARIATES:
    raw_smd = standardized_mean_difference(
        treatment[covariate],
        control_pool[covariate],
    )
    matched_smd = standardized_mean_difference(
        matched_pairs[f"treated_{covariate}"],
        matched_pairs[f"control_{covariate}"],
    )
    balance_records.append({
        "covariate": covariate,
        "raw_smd": raw_smd,
        "matched_smd": matched_smd,
        "raw_absolute_smd": abs(raw_smd),
        "matched_absolute_smd": abs(matched_smd),
        "matched_within_0_10": abs(matched_smd) <= 0.10,
    })

balance_table = (
    pd.DataFrame(balance_records)
    .sort_values("matched_absolute_smd", ascending=False)
    .set_index("covariate")
)

balance_table.round(3)

## Paired Campaign-Associated Lift

Each treatment date and its nearest control form one pair. The block bootstrap resamples
seven-day sequences within each year to retain short-term time dependence.

In [ ]:
paired_daily_aggregation = {
    "campaign_year": ("campaign_year", "first"),
}
for outcome in OUTCOMES:
    paired_daily_aggregation[f"treated_{outcome}"] = (
        f"treated_{outcome}",
        "first",
    )
    paired_daily_aggregation[f"control_{outcome}"] = (
        f"control_{outcome}",
        "mean",
    )

paired_daily = (
    matched_pairs.groupby("treated_date", as_index=False)
    .agg(**paired_daily_aggregation)
    .sort_values("treated_date")
)


# Purpose:
# Estimate a confidence interval while preserving weekly serial dependence.
#
# Used by:
# The paired demand, revenue, and ticket-yield lift table.
def moving_block_mean_interval(
    frame,
    value_column,
    block_length=7,
    bootstrap_samples=5000,
    seed=2026,
):
    rng = np.random.default_rng(seed)
    yearly_values = [
        group.sort_values("treated_date")[value_column].to_numpy()
        for _, group in frame.groupby("campaign_year")
    ]
    bootstrap_means = np.empty(bootstrap_samples, dtype=float)

    for sample_number in range(bootstrap_samples):
        resampled_values = []
        for values in yearly_values:
            required_blocks = int(np.ceil(len(values) / block_length))
            starts = rng.integers(0, len(values), size=required_blocks)
            indices = np.concatenate([
                (start + np.arange(block_length)) % len(values)
                for start in starts
            ])[: len(values)]
            resampled_values.extend(values[indices])
        bootstrap_means[sample_number] = np.mean(resampled_values)

    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


lift_records = []
for outcome in OUTCOMES:
    difference_column = f"{outcome}_difference"
    paired_daily[difference_column] = (
        paired_daily[f"treated_{outcome}"]
        - paired_daily[f"control_{outcome}"]
    )
    lower_95, upper_95 = moving_block_mean_interval(
        paired_daily, difference_column
    )
    mean_treated = paired_daily[f"treated_{outcome}"].mean()
    mean_control = paired_daily[f"control_{outcome}"].mean()
    mean_difference = paired_daily[difference_column].mean()
    difference_standard_deviation = paired_daily[
        difference_column
    ].std(ddof=1)
    lift_records.append({
        "outcome": outcome,
        "treatment_days": len(paired_daily),
        "treated_mean": mean_treated,
        "matched_control_mean": mean_control,
        "associated_lift": mean_difference,
        "associated_lift_pct": mean_difference / mean_control * 100,
        "lower_95": lower_95,
        "upper_95": upper_95,
        "paired_effect_size": (
            mean_difference / difference_standard_deviation
        ),
        "interval_excludes_zero": lower_95 * upper_95 > 0,
    })

matched_lift = pd.DataFrame(lift_records).set_index("outcome")
matched_lift.round(2)

## Regression Sensitivity and Pre-period Placebo

This second estimator uses all 1,096 historical days. It controls for other
campaigns, price, weather, holidays, school breaks, weekday, month, year, and
annual Fourier seasonality. HAC standard errors allow residual correlation across one week. The
28-day lead coefficient is a falsification check: a strong pre-launch shift
would weaken the campaign interpretation.

In [ ]:
daily_model = daily.copy()
for flag in (
    "holiday_flag",
    "school_break_flag",
    "severe_weather_flag",
    "discount_active",
    "paid_media_active",
    "bundle_active",
    "email_active",
):
    daily_model[flag] = daily_model[flag].astype(int)

daily_model["discount_lead_28d"] = False
for year in sorted(daily_model["year_number"].unique()):
    lead_start = pd.Timestamp(year=int(year), month=9, day=17)
    lead_end = pd.Timestamp(year=int(year), month=10, day=14)
    daily_model.loc[
        daily_model["calendar_date"].between(lead_start, lead_end),
        "discount_lead_28d",
    ] = True
daily_model["discount_lead_28d"] = daily_model[
    "discount_lead_28d"
].astype(int)

daily_model["day_of_year"] = daily_model[
    "calendar_date"
].dt.dayofyear
for harmonic in range(1, 5):
    angle = (
        2 * np.pi * harmonic * daily_model["day_of_year"] / 365.25
    )
    daily_model[f"annual_sin_{harmonic}"] = np.sin(angle)
    daily_model[f"annual_cos_{harmonic}"] = np.cos(angle)

seasonality_terms = " + ".join([
    f"annual_sin_{harmonic} + annual_cos_{harmonic}"
    for harmonic in range(1, 5)
])
control_terms = (
    "discount_active + discount_lead_28d + paid_media_active + "
    "bundle_active + email_active + planned_price_multiplier + "
    "holiday_flag + school_break_flag + avg_temperature_f + "
    "precipitation_in + severe_weather_flag + C(day_of_week) + "
    f"C(year_number) + {seasonality_terms}"
)
outcome_formulas = {
    "net_revenue": f"np.log(net_revenue) ~ {control_terms}",
    "net_demand": f"np.log(net_demand) ~ {control_terms}",
    "net_revenue_per_ticket": (
        f"np.log(net_revenue_per_ticket) ~ {control_terms}"
    ),
}

regression_models = {}
regression_records = []
for outcome, formula in outcome_formulas.items():
    fitted = smf.ols(formula, data=daily_model).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 7},
    )
    regression_models[outcome] = fitted
    design_rank = np.linalg.matrix_rank(fitted.model.exog)
    if design_rank != fitted.model.exog.shape[1]:
        raise ValueError(f"Rank-deficient regression for {outcome}.")

    for term, period in (
        ("discount_active", "campaign_window"),
        ("discount_lead_28d", "pre_period_placebo"),
    ):
        lower_log, upper_log = fitted.conf_int().loc[term]
        regression_records.append({
            "outcome": outcome,
            "period": period,
            "observations": int(fitted.nobs),
            "associated_change_pct": np.expm1(fitted.params[term]) * 100,
            "lower_95_pct": np.expm1(lower_log) * 100,
            "upper_95_pct": np.expm1(upper_log) * 100,
            "p_value": fitted.pvalues[term],
        })

regression_results = (
    pd.DataFrame(regression_records)
    .set_index(["outcome", "period"])
)
regression_results.round(3)

## Commercial Interpretation

Daily net revenue already subtracts ticket discounts and refunds, but not
marketing spend. The calculation below compares matched incremental net
revenue with spend during the clean treatment dates. Campaign delivery uses
purchase dates while lift uses visit dates, so this is a planning estimate,
not an audited profit statement.

In [ ]:
clean_treatment_dates = set(treatment["calendar_date"])
clean_campaign_costs = campaign_costs.loc[
    campaign_costs["calendar_date"].isin(clean_treatment_dates)
]

daily_revenue_lift = matched_lift.loc["net_revenue", "associated_lift"]
revenue_lower = matched_lift.loc["net_revenue", "lower_95"]
revenue_upper = matched_lift.loc["net_revenue", "upper_95"]
treatment_days = len(paired_daily)
estimated_incremental_net_revenue = daily_revenue_lift * treatment_days
marketing_spend = clean_campaign_costs["marketing_spend"].sum()

commercial_summary = pd.Series({
    "clean_treatment_days": treatment_days,
    "estimated_incremental_net_revenue": (
        estimated_incremental_net_revenue
    ),
    "incremental_revenue_lower_95": revenue_lower * treatment_days,
    "incremental_revenue_upper_95": revenue_upper * treatment_days,
    "marketing_spend": marketing_spend,
    "attributed_discount_amount": clean_campaign_costs[
        "attributed_discount_amount"
    ].sum(),
    "estimated_net_revenue_after_marketing": (
        estimated_incremental_net_revenue - marketing_spend
    ),
    "associated_revenue_on_marketing_spend": (
        estimated_incremental_net_revenue / marketing_spend
    ),
    "associated_return_after_marketing_spend": (
        (estimated_incremental_net_revenue - marketing_spend)
        / marketing_spend
    ),
})

commercial_summary.round(2).to_frame("value")

In [ ]:
primary_result = matched_lift.loc["net_revenue"]
yield_result = matched_lift.loc["net_revenue_per_ticket"]
regression_revenue_result = regression_results.loc[
    ("net_revenue", "campaign_window")
]
regression_yield_result = regression_results.loc[
    ("net_revenue_per_ticket", "campaign_window")
]

if primary_result["lower_95"] > 0:
    revenue_evidence = "positive_associated_net_revenue_lift"
elif primary_result["upper_95"] < 0:
    revenue_evidence = "negative_associated_net_revenue_lift"
else:
    revenue_evidence = "inconclusive_net_revenue_lift"

if (
    yield_result["upper_95"] < 0
    and regression_yield_result["upper_95_pct"] < 0
):
    yield_evidence = "consistent_lower_net_revenue_per_ticket"
elif (
    yield_result["lower_95"] > 0
    and regression_yield_result["lower_95_pct"] > 0
):
    yield_evidence = "consistent_higher_net_revenue_per_ticket"
elif (
    yield_result["interval_excludes_zero"]
    or regression_yield_result["p_value"] < 0.05
):
    yield_evidence = "mixed_ticket_yield_evidence"
else:
    yield_evidence = "inconclusive_ticket_yield_change"

decision_summary = pd.Series({
    "primary_revenue_evidence": revenue_evidence,
    "regression_revenue_evidence": (
        "positive_associated_net_revenue_lift"
        if regression_revenue_result["lower_95_pct"] > 0
        else "not_consistently_positive"
    ),
    "ticket_yield_evidence": yield_evidence,
    "matched_balance_passes_all_0_10": bool(
        balance_table["matched_within_0_10"].all()
    ),
    "pre_period_revenue_placebo_p_value": regression_results.loc[
        ("net_revenue", "pre_period_placebo"),
        "p_value",
    ],
    "pre_period_revenue_placebo_passes_0_05": bool(
        regression_results.loc[
            ("net_revenue", "pre_period_placebo"),
            "p_value",
        ] >= 0.05
    ),
    "causal_claim_allowed": False,
})

decision_summary.to_frame("result")

## Interpretation Contract

A commercially favorable result requires more than attributed revenue: matched
net revenue should increase, its interval should be reasonably separated from
zero, and the gain should cover marketing spend. A lower ticket yield does not
automatically mean failure if incremental demand creates enough total net
revenue. Conversely, attributed ROAS alone cannot prove incrementality because
some attributed customers may have purchased without the campaign.

Even if balance, placebo, matching, and regression results agree, unmeasured
confounding may remain. A future production experiment should randomize eligible
customers, markets, or dates and preserve assignment and exposure records.